# 📊 Unemployment Analysis with Python
### CodeAlpha Data Science Internship — Task 2

**Objective:** Analyze the impact of the COVID-19 pandemic and subsequent nationwide lockdowns on unemployment rates across rural and urban India using Python, Pandas, and Seaborn.

## 1. Import Libraries & Set Plotting Styles

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Data Loading & Inspection

In [ ]:
data_path = os.path.join('..', 'data', 'Unemployment in India.csv')
df = pd.read_csv(data_path)
print(f'Shape of dataset: {df.shape}')
df.head()

## 3. Data Cleaning & Preprocessing
- Strip leading and trailing whitespaces from headers and categorical strings.
- Drop completely empty rows (`how='all'`).
- Convert date strings to datetime objects and extract Year/Month features.

In [ ]:
# Clean column headers
df.columns = df.columns.str.strip()

# Drop 28 blank rows
df = df.dropna(how='all')

# Strip text values
df['Frequency'] = df['Frequency'].str.strip()
df['Area'] = df['Area'].str.strip()
df['Region'] = df['Region'].str.strip()

# Parse dates
df['Date'] = pd.to_datetime(df['Date'].str.strip(), dayfirst=True)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()

# Tag Pre vs Post Lockdown period
df['Period'] = df['Date'].apply(
    lambda d: 'Lockdown (Apr-Jun 2020)' if d >= pd.to_datetime('2020-04-01') else 'Pre-Lockdown (May 2019-Mar 2020)'
)

print(f'Cleaned Shape: {df.shape}')
df.info()

## 4. Exploratory Data Analysis & Visualizations
### 4.1 Timeline Analysis: Rural vs Urban Disparity

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Date', y='Estimated Unemployment Rate (%)', hue='Area', marker='o', ci=None)
plt.axvline(pd.to_datetime('2020-03-24'), color='red', linestyle='--', linewidth=2, label='Lockdown Imposed (24 Mar 2020)')
plt.title('Unemployment Rate Timeline in India (Rural vs Urban)', fontsize=14, fontweight='bold')
plt.xlabel('Timeline', fontsize=12)
plt.ylabel('Unemployment Rate (%)', fontsize=12)
plt.legend(title='Area / Event')
plt.tight_layout()
plt.show()

### 4.2 Pre-Lockdown vs. Lockdown Distribution Shift

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Period', y='Estimated Unemployment Rate (%)', palette='Set2')
plt.title('Pre vs Lockdown Unemployment Rate Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Economic Period', fontsize=11)
plt.ylabel('Unemployment Rate (%)', fontsize=11)
plt.tight_layout()
plt.show()

### 4.3 Top 10 Hardest-Hit States During Lockdown

In [ ]:
lockdown_df = df[df['Date'] >= pd.to_datetime('2020-04-01')]
top_states = (
    lockdown_df.groupby('Region')['Estimated Unemployment Rate (%)']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_states, y='Region', x='Estimated Unemployment Rate (%)', palette='Reds_r')
plt.title('Top 10 States with Highest Unemployment (Apr-Jun 2020)', fontsize=13, fontweight='bold')
plt.xlabel('Average Unemployment Rate (%)', fontsize=11)
plt.ylabel('State / Region', fontsize=11)
plt.tight_layout()
plt.show()

### 4.4 Correlation Analysis

In [ ]:
plt.figure(figsize=(8, 6))
numeric_cols = [
    'Estimated Unemployment Rate (%)',
    'Estimated Employed',
    'Estimated Labour Participation Rate (%)'
]
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Quantitative Summary & Key Findings
1. **Average Rate Spike**: The mean national unemployment rate jumped from **9.61%** pre-lockdown to **20.19%** during the lockdown phase (+110.03% increase).
2. **Urban Disparity**: Urban areas reached a higher peak (**22.08%**) compared to rural areas (**18.26%**) due to mobility and industrial restrictions.
3. **State Disparities**: Puducherry, Jharkhand, Haryana, and Bihar suffered the most severe spikes during April–June 2020.